# Esperimento di Topic Labeling

In questo notebook confrontiamo diversi approcci per l'etichettatura automatica dei cluster individuati tramite HDBSCAN.

## Obiettivi:
1. Caricare i cluster e i documenti associati.
2. Applicare **YAKE**, **TextRank**, **c-TF-IDF** e **KeyBERT (Simulated)**.
3. Estrarre i documenti rappresentativi per ogni cluster.
4. Salvare i risultati comparativi nei metadata.

In [1]:
import pandas as pd
import numpy as np
import os
import sys
import json
import re
from sentence_transformers import SentenceTransformer

# Aggiungiamo src al path per importare le utility
sys.path.append(os.path.abspath("../../"))

from src.utils.topic_labeling import (
    extract_keywords_yake, 
    extract_keywords_textrank, 
    calculate_ctfidf, 
    extract_keywords_keybert,
    get_cluster_representative_docs
)

## 1. Caricamento Dati

In [2]:
df_processed = pd.read_parquet("../../data/processed/jmail_emails_processed.parquet")
df_clusters = pd.read_parquet("../../data/processed/email_cluster_assignments.parquet")
embeddings = np.load("../../data/embeddings/email_embeddings_bge-small-en-v1-5.npy")

df = df_processed.merge(df_clusters, on="id", how="inner")
print(f"Totale righe: {len(df)}")

Totale righe: 42471


## 2. Applicazione Multi-Algoritmo
Eseguiamo tutti i metodi per ogni cluster.

In [3]:
from tqdm.notebook import tqdm

# Carichiamo il modello di embedding per KeyBERT
print("Caricamento del modello di embedding...")
embedding_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

docs_per_cluster = df.groupby("cluster_id")["combined_text"].apply(lambda x: " ".join(x)).to_dict()
cluster_list = sorted([c for c in docs_per_cluster.keys()])

# Pre-calcolo c-TF-IDF
print("Calcolo c-TF-IDF globale...")
ctfidf_labels_dict = calculate_ctfidf(docs_per_cluster, top_n=20)

final_labels = {}

print("Inizio estrazione labels per ogni cluster...")
for cid in tqdm(cluster_list):
    cluster_df = df[df["cluster_id"] == cid]
    
    # 1. Troviamo i documenti più rappresentativi per il sample text
    cluster_indices = cluster_df["embedding_row"].values
    cluster_embs = embeddings[cluster_indices]
    cluster_emb_centroid = cluster_embs.mean(axis=0)
    
    rep_docs = get_cluster_representative_docs(cluster_df["combined_text"].tolist(), cluster_embs, cluster_emb_centroid, n=50)
    sample_text = " ".join(rep_docs)
    
    # 2. Generiamo i veri Word Embeddings per KeyBERT
    words_in_sample = list(set(re.findall(r"\b[a-z]{3,}\b", sample_text.lower())))
    if len(words_in_sample) > 300:
        words_in_sample = words_in_sample[:300]
        
    # Se non ci sono parole valide (es. cluster di rumore senza testo utile)
    if not words_in_sample:
        final_labels[str(cid)] = {
            "ctfidf": ctfidf_labels_dict.get(cid, []),
            "yake": [],
            "textrank": [],
            "keybert_sim": []
        }
        continue
        
    word_embs_array = embedding_model.encode(words_in_sample, show_progress_bar=False)
    real_word_embs = {w: emb for w, emb in zip(words_in_sample, word_embs_array)}
    
    # 3. Applichiamo tutti gli algoritmi
    final_labels[str(cid)] = {
        "ctfidf": ctfidf_labels_dict.get(cid, []),
        "yake": extract_keywords_yake(sample_text, top_n=20),
        "textrank": extract_keywords_textrank(sample_text, top_n=20),
        "keybert_sim": extract_keywords_keybert(rep_docs[:10], cluster_emb_centroid, real_word_embs, top_n=20)
    }

print("Labeling completato.")

Caricamento del modello di embedding...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Calcolo c-TF-IDF globale...
Inizio estrazione labels per ogni cluster...


  0%|          | 0/13 [00:00<?, ?it/s]

/home/filippo/Scrivania/pizza-cluster/.venv/lib/python3.12/site-packages
Labeling completato.


## 3. Salvataggio Metadata

In [4]:
output_path = "../../data/metadata/cluster_labeling_metadata.json"
metadata = {
    "generated_at": pd.Timestamp.now().isoformat(),
    "algorithms": ["c-TF-IDF", "YAKE", "TextRank", "KeyBERT-Sim"],
    "n_clusters": len(final_labels),
    "cluster_labels": final_labels
}

with open(output_path, "w") as f:
    json.dump(metadata, f, indent=4)
print(f"Metadata aggiornati in {output_path}")

Metadata aggiornati in ../../data/metadata/cluster_labeling_metadata.json


## 4. Visualizzazione Comparativa

In [5]:
comparison_df = []
for cid in final_labels.keys():
    row = {"cluster": cid}
    row.update(final_labels[cid])
    comparison_df.append(row)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
display(pd.DataFrame(comparison_df))


,cluster,ctfidf,yake,textrank,keybert_sim
0,-1,"[deutsche, td, bank, communication, com, confidential, new, epstein, york, db, yahoo, information, client, wealth, jeffrey, email, trust, avenue, ny, account]","[Subject, Bank, Deutsche, communication, York, Kahn, Confidential, Richard, Jeffrey, Classification, Trust, Avenue, information, Financial, email, Epstein, Nina, Company, KYC, management]","[| | | |Bergander | | | | | | Sent, | | | |, | | | | work, | | |, | | | | |Warner, | |, | Ticker | | Price, Client Contact | | | Richard Kahn, | | cell, | | |sent, | |-|-| |, | Beneficiary | Amount | |---|---|, Close | | Price, Retrophin | | RTRX, | | DB GMR Target, | | | |-------+-----+-----------+------+----------+----+-------+--------+------+ |STREAM, Deutsche Bank Trust Company Americas Deutsche Bank Wealth Management, | NY |TBD|, Wealth Management Americas Deutsche Bank Wealth Management Deutsche Bank Trust Company, Wealth Management Americas Deutsche Bank Wealth Management Deutsche Bank Trust Company Americas]","[answers, responding, repaid, hsbc, brokerage, submitted, issuer, statements, broker, complimentary, payout, send, put, given, refreshed, withholding, sent, back, specified, flagging]"
1,0,"[times, newyorktimesinfo, com, new, nytimes, york, http, digital, offer, subscription, weeks, 8041, privacy, nyt, delivery, apps, email, unlimited, access, reader]","[Times, YORK, Digital, Offer, Subscription, time, Special, weeks, NYT, REDEEM, delivery, unlimited, access, apps, subscriptions, View, Unsubscribe, Davenport, Company, Crossword]","[New York Times, New York Times digital subscriptions, New York Times Crossword, Times subscription, Times subscriptions, Times Digital Subscription, Times Subscription, Times, Times readers, Times Premier content, Times Gift, Dear Times Reader, REDEEM SPECIAL OFFER, New NYT Opinion, The New York Times http://e.newyorktimesinfo.com/a/tBTcd8TB81HXJB85-jTAAwENc9d/nyt Privacy Policy, TODAY REDEEM SPECIAL OFFER, The New York Times Company P.O. Box, The New York Times Crosswords apps, REDEEM OFFER, The New York Times delivery service]","[nyt, newyorktimesinfo, subscribe, subscription, unsubscribe, subscriber, times, columnists, coupons, paypal, plus, journalists, reader, cents, print, email, journalism, daily, articles, savings]"
2,1,"[wikisource, senator, alberto, statement, gonzales, barack, library, nomination, attorney, general, obama, online, floor, free, communication, thereof, mail, use, information, reserved]","[Wikisource, Floor, General, Statement, Senator, Barack, Obama, Nomination, Alberto, Gonzales, Attorney, free, online, library, Wow, Date, Furnished, Number, Mid-October, reconsider]","[Senator Barack Obama, Alberto Gonzales, Barack Obama, Floor Statement, the free online library, Mid-October, Attorney General - Wikisource, the free online library\n\nDate, the free online library\n\nNumber, the free online library\n\neric, General - Wikisource, October, the Nomination of Alberto Gonzales, Mid, the Nomination, Re, eric, un, Yep, another run]","[obama, barack, gonzales, nomination, senator, statement, attorney, floor, whose, october, reconsider, for, wikisource, general, eric, him, the, alberto, apt, free]"
3,2,"[case, assigned, salesforce, assignment, notification, annual, link, description, click, customer, access, kyc, https, following, type, confidential, contact, note, new, dbforcepb]","[Case, Jeffrey, Epstein, CONFIDENTIAL, KYC, Description, ASSIGNMENT, Annual, Call, Note, Subject, Type, NOTIFICATION, Contact, Click, assigned, PURSUANT, FED, CRIM, Customer]","[Case Subject, NEW CASE, Case #, Jeffrey Epstein case, Case, Case tt, Jeffrey E. Epstein Case, https://na4.salesforce.com/5006000000VDvNf Case #, Case U, https:/ Case, Case A, Case I, O44 NEW CASE, EFTA01400360 Case, EFTA01400339 Case #, EFTA01404107 Case #, EFTA01404404 Case #, https://dbforcepb.my.salesforce.com/5003200001DDBDG EFTA01415174 Case, https://na4.salesforce.com/5006000000VDvNf EFTA01404975 Case, https://na4.sales